# Chapter 9: Valuation Adjustments (XVAs)
## Demystifying CVA, DVA, FVA, MVA, and KVA with Python

Imagine you lend $\$100$ to your reliable cousin, and $\$100$ to your eccentric friend who has a habit of disappearing. Even though both loans have a nominal value of $\$100$, you naturally value the loan to your friend at a discount because there's a real chance you won't see that money again. 

In derivatives markets, this common-sense adjustment is called a **Valuation Adjustment (XVA)**. 

Since the 2008 global financial crisis, derivatives valuation is no longer just about the "mid-market" price of the contract under no-default assumptions ($f_{nd}$). Instead, banks must adjust the value of their portfolios to reflect **counterparty credit risk**, **funding costs**, **collateral margin requirements**, and **regulatory capital charges**.

In this notebook, we will explore **Chapter 9** of John Hull's *Options, Futures, and Other Derivatives*, covering:
1. **CVA & DVA**: Adjustments for the default risk of the counterparty and the bank itself.
2. **FVA & MVA**: Adjustments for the funding of variation margin and initial margin.
3. **KVA**: Capital valuation adjustments for regulatory capital requirements.
4. **Practice Questions**: Detailed analytical and programmatic Python solutions to **Practice Questions 9.1 to 9.8**.


---
## 1. Credit Valuation Adjustment (CVA) & Debit Valuation Adjustment (DVA)

### Credit Valuation Adjustment (CVA)
**CVA** is the bank's estimate of the present value of the expected cost of a counterparty defaulting.
Suppose a bank has a portfolio of derivatives with a client. If the client defaults when the portfolio has a positive value to the bank (it's an asset), the bank will lose some or all of that value.

Mathematically, the bank divides the life of the portfolio ($T$ years) into $N$ intervals. For each interval $i$:
* $q_i$: the probability of default by the counterparty during interval $i$.
* $v_i$: the present value of the expected loss if there is a default at the midpoint of interval $i$.
$$CVA = \sum_{i=1}^N q_i v_i$$

Taking counterparty risk into account, the portfolio's adjusted value to the bank is:
$$\text{Adjusted Value} = f_{nd} - CVA$$

---

### Debit Valuation Adjustment (DVA)
But wait—what if the *bank itself* defaults? 
If the bank defaults when the derivatives portfolio has a **negative** value to the bank (meaning it's a liability, and the bank owes money to the counterparty), the bank avoids paying its obligation. This represents an expected gain to the bank's creditors!

**DVA** is the present value of the expected gain to the bank from the possibility that it might default:
$$DVA = \sum_{i=1}^N q^*_i v^*_i$$

Where:
* $q^*_i$: the probability of a default by the bank during interval $i$.
* $v^*_i$: the present value of the bank's gain (and counterparty's loss) if the bank defaults at the midpoint.

Taking both CVA and DVA into account, the total value of the portfolio is:
$$\text{Adjusted Value} = f_{nd} - CVA + DVA$$

> [!CAUTION]
> ### The DVA Paradox: Worsening Credit is "Good"?
> DVA has a highly counterintuitive effect: **as a bank's creditworthiness declines, its DVA increases, making its derivatives portfolio appear more valuable!** 
> 
> During the financial crisis, several major banks recorded massive accounting profits simply because their own credit spreads widened, which pushed up their DVA. While economically correct (they were less likely to pay their liabilities), it represents a paper gain that cannot be spent or easily hedged.


---
## 2. Funding Valuation Adjustment (FVA) & Margin Valuation Adjustment (MVA)

### Funding Valuation Adjustment (FVA)
When a bank enters into an uncollateralized derivative (like a swap with a corporate end user) and hedges it with a collateralized derivative (like a swap cleared through a central clearinghouse - CCP), it faces funding requirements:

* **Funding Cost (FCA)**: If the uncollateralized swap has a positive value (an asset), the hedging trade with the CCP has a negative value. The bank must post **variation margin (VM)** cash to the CCP but receives no collateral from the corporate client. The bank has to borrow these funds from external sources, incurring a funding cost.
* **Funding Benefit (FBA)**: If the uncollateralized swap has a negative value (a liability), the hedging trade with the CCP has a positive value. The bank receives VM cash from the CCP but does not have to post any collateral to the client. This is a free source of funding, reducing the bank's need for external borrowing.

$$FVA = FCA - FBA$$

---

### Margin Valuation Adjustment (MVA)
Modern regulations require banks to post **Initial Margin (IM)** to clearinghouses or bilateral counterparties to cover potential future wiggles in the portfolio's value in a default scenario. 
Posting this collateral requires funding. The present value of the expected cost of funding this initial margin over the life of the trade is the **Margin Valuation Adjustment (MVA)**.


---
## 3. The Great XVA Debate: Economists vs. Financial Engineers

There is a fascinating, ongoing debate in quantitative finance between **Financial Economists** and **Practitioners (Financial Engineers)** on how FVA, MVA, and KVA should be calculated. 

This is essentially a debate between **Marginal Costs (Risk-Adjusted)** and **Average Costs (Hurdle Rates)**.

### The Practitioner's View (The Derivatives Desk):
*"If the bank's treasury borrows funds at an average rate of $\text{Fed Funds} + 100\text{ bps}$, and we tie up cash in CCP margin, we are losing $100\text{ bps}$ per year. We must pass this funding cost directly to the derivatives pricing (FVA/MVA) using our **average debt funding cost**."*

### The Economist's View (Modigliani-Miller & Corporate Finance):
*"The hurdle rate for an investment must reflect the **risk of the investment itself**, not the cost of the bank's overall debt. Initial and variation margins posted to a CCP are virtually risk-free. Therefore, the required return on that cash should be close to the **risk-free rate** (marginal cost), not the bank's risky corporate debt cost."*

If a bank double in size by undertaking only low-risk, risk-free projects, its overall corporate risk decreases, and the market will automatically lower its average funding costs. Charging a high FVA/MVA based on average costs makes low-risk, safe projects appear unattractive and high-risk projects appear attractive, ultimately making the bank more risky over time.


---
# Part 2: Chapter 9 Practice Questions & Python Solutions

Let's dive into the **Practice Questions 9.1 to 9.8** from the textbook to master these concepts mathematically and logically. We'll implement the quantitative calculations directly in Python.


---
## Practice Question 9.1

### Question
Explain the difference between the views of financial economists and most practitioners on how MVA and FVA should be calculated.

### Solution
* **Practitioners (The Bankers)**:
 * They focus on the physical cost of funding. If they have to post collateral (variation margin for FVA, or initial margin for MVA) to a CCP, they fund this cash by borrowing from the bank's treasury.
 * The bank's treasury raises funds at its **average corporate debt cost** (e.g. $\text{Fed Funds} + 100\text{ bps}$).
 * Therefore, practitioners calculate FVA and MVA by discounting expected future margin requirements using the bank's average corporate borrowing spread.
* **Financial Economists**:
 * They apply classical corporate finance principles: the discount rate of a project must reflect the **risk of that specific project**, not how the firm is financed.
 * Cash posted as initial or variation margin at a CCP is highly secure and virtually **risk-free**.
 * Therefore, the required rate of return on this margin should be close to the **risk-free rate** (marginal cost), not the bank's risky borrowing cost.
 * Using average corporate funding costs to evaluate low-risk margin requirements creates a distortion, making low-risk trades seem artificially expensive.


---
## Practice Question 9.2

### Question
Explain the difference between the views of financial economists and most practitioners on how KVA should be calculated.

### Solution
* **Practitioners**:
 * Regulatory capital rules (like Basel III) require banks to hold equity capital against derivatives exposure.
 * If a new trade requires holding an additional $1 million of equity capital, and the bank's shareholders demand an average return of $15\%$ per annum on their equity, practitioners argue the trade must charge a **Capital Valuation Adjustment (KVA)** to ensure that the incremental capital earns at least a $15\%$ return.
* **Financial Economists**:
 * They base their argument on the **Modigliani-Miller theorem**: how a company is financed does not affect its enterprise value or risk-adjusted project hurdles.
 * As the bank holds more equity capital relative to debt, its leverage decreases, making the bank **safer** (less risky).
 * Consequently, because the bank is less risky, shareholders will naturally demand a **lower return** on their equity.
 * Therefore, the marginal return required on the new, safer equity capital is much lower than the bank's historical average hurdle rate of $15\%$. Economists argue that charging a high KVA based on the average $15\%$ cost is theoretically incorrect.


---
## Practice Question 9.3

### Question
Explain why FVA can be calculated for a transaction without considering the portfolio to which the transaction belongs, but that the same is not true of MVA.

### Solution
* **FVA (Funding Valuation Adjustment)**:
 * FVA arises from funding **variation margin (VM)**. Variation margin directly tracks the daily market value wiggles of each individual derivative trade.
 * Since derivatives values are **linear and additive**, the net variation margin funding needed for a portfolio of trades is simply the sum of the variation margins of the individual transactions:
 $$VM_{\text{portfolio}} = \sum VM_j$$
 * Thus, we can calculate FVA on a transaction-by-transaction basis in isolation.
* **MVA (Margin Valuation Adjustment)**:
 * MVA arises from funding **initial margin (IM)**. Initial margin is calculated by CCPs or bilateral dealers on a **portfolio basis** using non-linear risk measures (like Value at Risk, VaR).
 * Because of diversification, netting, and hedging, a new trade might offset existing risks in the portfolio, actually **reducing** the overall portfolio initial margin.
 * Since IM is highly non-linear and depends entirely on what other trades are in the portfolio, MVA **cannot** be calculated in isolation for a single transaction. We must model the entire portfolio to determine the new trade's incremental MVA.


---
## Practice Question 9.4

### Question
Suppose that a bank buys an option from a client. The option is uncollateralized and there are no other transactions outstanding with the client. The expected values of the option at the midpoint of years 1, 2, and 3 are 6, 5, and 4. The probability of the counterparty defaulting in each of the three years is 3%. The probability of the bank defaulting in each of the three years is 2%. Estimate the bank’s CVA and DVA for the transaction. Assume no recovery in the event of a default and zero interest rates.

### Solution
1. **Analyze CVA**:
 * The bank **bought the option**, so the option is an asset to the bank. Its expected values are positive ($+6, +5, +4$).
 * The bank is exposed to the counterparty's default.
 * Default probability per interval $q_i = 3\% = 0.03$.
 * Recovery rate $R = 0$ (so expected loss is $100\%$ of the value).
 * Since interest rates are $0\%$, the discount factors are $DF_i = 1.0$.
 $$CVA = \sum_{i=1}^3 q_i v_i = (0.03 \times 6) + (0.03 \times 5) + (0.03 \times 4) = 0.03 \times (6 + 5 + 4) = 0.45$$
2. **Analyze DVA**:
 * Since the bank bought the option, the contract is an asset to the bank and can **never have a negative value** to the bank (the bank has no future liabilities or payments under the option contract).
 * The counterparty is never exposed to the bank's default. Therefore, the present value of the bank's gain from its own default is zero at every midpoint ($v^*_i = 0$).
 * **DVA = 0**.

Let's use Python to verify these numbers!


In [1]:
# --- Question 9.4 Calculations ---
import numpy as np

expected_values = np.array([6.0, 5.0, 4.0]) # midpoints of Year 1, 2, 3
q_client = 0.03 # Client default probability per year
q_bank = 0.02 # Bank default probability per year
recovery_rate = 0.0 # 0% recovery

# Since bank bought the option, its value is always positive (asset)
v_i = expected_values # Client exposure
v_star_i = np.zeros_like(expected_values) # Bank exposure is zero (option cannot be negative to bank)

# CVA and DVA
cva = (q_client * (1 - recovery_rate) * v_i).sum()
dva = (q_bank * (1 - recovery_rate) * v_star_i).sum()

print(f"Credit Valuation Adjustment (CVA): {cva:.4f}")
print(f"Debit Valuation Adjustment (DVA): {dva:.4f}")


Credit Valuation Adjustment (CVA): 0.4500
Debit Valuation Adjustment (DVA): 0.0000


---
## Practice Question 9.5

### Question
“The impact of DVA on earnings volatility is generally greater than that of CVA.” Explain this statement.

### Solution
This statement is true due to the **hedging mismatch** between CVA and DVA:
1. **Hedging CVA**: 
 * CVA represents the default risk of the bank's counterparties.
 * If a counterparty is a publicly-traded firm, the bank can **actively hedge CVA** by buying Credit Default Swaps (CDS) or shorting the bond/equity of that counterparty. If the counterparty's credit rating deteriorates, the CDS gains offset the CVA losses, stabilizing earnings.
2. **Hedging DVA**:
 * DVA represents the bank's *own credit risk*.
 * If the bank's own credit spread widens (meaning its creditworthiness deteriorates), its DVA increases, creating an accounting gain in earnings.
 * However, **a bank cannot buy a CDS protecting against its own default** (that contract would be worthless if the bank actually defaulted). 
 * Because it is legally and practically impossible for a bank to hedge its own credit spread wiggles, DVA represents a massive, completely **unhedged exposure** on the bank's balance sheet. When market spreads fluctuate, DVA paper gains/losses swing wildly, causing immense earnings volatility.


---
## Practice Question 9.6

### Question
A company is trying to decide between issuing debt and equity to fulfill a funding need. What in theory should happen to the return required by equity holders if it chooses (a) debt and (b) equity?

### Solution
1. **(a) If the company chooses Debt**:
 * Issuing debt increases the company's financial leverage (gearing).
 * Debt has senior claims on earnings. This increases the volatility of the residual earnings available to equity holders and raises the probability of financial distress/bankruptcy.
 * Because equity holders are now taking on significantly more risk, they will demand a **higher rate of return** (the cost of equity capital increases).
2. **(b) If the company chooses Equity**:
 * Issuing new equity reduces the company's financial leverage.
 * The company has less debt interest to pay, making it safer and more resilient to business shocks.
 * Because equity holders face less risk of financial distress, their required rate of return will **decrease** (the cost of equity capital decreases).


---
## Practice Question 9.7

### Question
Explain the meaning of “netting”. Suppose no collateral is posted. Why does a netting agreement usually reduce credit risks to both sides? Under what circumstances does netting have no effect on credit risk?

### Solution
* **Meaning of Netting**:
 * Netting is a legal agreement (often formalized under an ISDA Master Agreement) specifying that if one party defaults, all outstanding derivatives contracts between the two parties are terminated.
 * Their positive and negative market values are summed together, and only a **single, net settlement payment** is made by one party to the other.
* **Why Netting Reduces Credit Risk**:
 * Without netting, the defaulting party could "cherry-pick" the trades: they would demand full payment for all contracts that have a positive value to them, while defaulting on their obligations for the contracts that have a negative value to them.
 * Netting legally prevents this. If Bank A has one trade worth $+\$15\text{M}$ and another worth $-\$10\text{M}$ with Bank B, netting reduces Bank A's exposure to only **$+\$5\text{M}$** (instead of the $+\$15\text{M}$ risk it would carry without netting).
* **When Netting has No Effect**:
 * Netting has no effect if **all** outstanding transactions between the two parties are either entirely positive for one side (and negative for the other). In that case, there are no offsetting negative contracts to reduce the exposure.
 * Similarly, netting has no effect if there is only a **single transaction** outstanding between the counterparties.


---
## Practice Question 9.8

### Question
The average funding cost for a company is 5% per annum when the risk-free rate is 3%. The company is currently undertaking projects worth $9 million. It plans to increase its size by undertaking $1 million of risk-free projects. What would you expect to happen to its average funding cost?

### Solution
1. **Analyze existing risk**:
 * The company has $\$9\text{M}$ of existing corporate projects.
 * Its average funding cost is $5.0\%$.
 * Since the risk-free rate is $3.0\%$, the company's credit/risk spread is $5\% - 3\% = 2\%$.
2. **Analyze new risk**:
 * The company adds $\$1\text{M}$ of new projects that are **entirely risk-free**.
 * Under financial economic theory, the cost of funding these new risk-free projects is exactly the risk-free rate: **$3.0\%$** (marginal funding cost).
3. **Calculate new average cost**:
 * The new total size of the company is $\$10\text{M}$.
 * The new expected average funding cost is the weighted average of the old projects and the new risk-free projects:
 $$\text{New Average Cost} = \frac{\$9\text{M}}{\$10\text{M}} \times 5.0\% + \frac{\$1\text{M}}{\$10\text{M}} \times 3.0\% = 4.5\% + 0.3\% = \mathbf{4.80\%}$$
 * The company's average funding cost will decrease from $5.0\%$ to **$4.80\%$** because it diluted its risk profile by adding safe assets.

Let's write Python code to calculate this!


In [2]:
# --- Question 9.8 Calculations ---
old_size = 9.0 # $9 Million
old_cost = 0.05 # 5.0% average cost
new_proj_size = 1.0 # $1 Million risk-free project
risk_free_rate = 0.03 # 3.0% risk-free rate

total_size = old_size + new_proj_size

# Weighted average cost of capital
new_average_cost = (old_size / total_size) * old_cost + (new_proj_size / total_size) * risk_free_rate

print(f"Old Average Funding Cost: {old_cost*100:.2f}%")
print(f"New Average Funding Cost: {new_average_cost*100:.2f}%")
print(f"Reduction in Funding Cost: {(old_cost - new_average_cost)*100*100:.1f} basis points")


Old Average Funding Cost: 5.00%
New Average Funding Cost: 4.80%
Reduction in Funding Cost: 20.0 basis points
